In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, input_file_name, to_timestamp, to_date, date_sub, current_date, row_number
from pyspark.sql.window import Window

if "PYSPARK_SUBMIT_ARGS" in os.environ:
    del os.environ["PYSPARK_SUBMIT_ARGS"]

packages = "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.6.0"
spark = SparkSession.builder \
    .appName("DailyTransactionPipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", packages) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "matrix") \
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Spark Version: 3.5.2


In [42]:
yesterday_str = date_sub(current_date(),10).cast("string")
#ayın 19-u və 20-i seçilməlidi, çünki verilən data o vaxta uyğundu. səhv seçsək əlimizə data gəlməyəcək.

In [43]:
df_atm = spark.read.csv("s3a://bronze/landing/atm/", header=True, inferSchema=True) \
    .filter(col("tx_date") == yesterday_str) \
    .withColumn("channel", lit("ATM")) \
    .withColumn("source_file", input_file_name())

In [44]:
df_atm.show(5)

+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----------+-------+--------------------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|   tx_date|channel|         source_file|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----------+-------+--------------------+
|TX7979444711|2026-02-19 08:36:00|CARD300007|6011|  -5.0|     AZN|    DEU|  Munich|   ATM37384|2026-02-19|    ATM|s3a://bronze/land...|
|TX5066159704|2026-02-19 16:24:00|CARD300084|6011|484.59|     AZN|    TUR|   Izmir|   ATM52352|2026-02-19|    ATM|s3a://bronze/land...|
|TX4054997588|2026-02-19 19:19:00|CARD300021|6011|244.96|     TRY|    USA|   Miami|   ATM13474|2026-02-19|    ATM|s3a://bronze/land...|
|TX9784851860|2026-02-19 22:30:00|CARD300005|6011|449.11|     TRY|    AZE|    Baku|   ATM08949|2026-02-19|    ATM|s3a://bronze/land...|
|TX5668410258|2026-02-19 19:46:00|CARD300148|601

In [45]:
df_ecom = spark.read.csv("s3a://bronze/landing/ecom/", header=True, inferSchema=True) \
    .filter(col("tx_date") == yesterday_str) \
    .withColumn("channel", lit("ECOM")) \
    .withColumn("source_file", input_file_name())

In [48]:
df_ecom.show(5)

+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----------+-------+--------------------+
|       tx_id|              tx_ts|   card_no|customer_no| mcc|amount|currency|country|    city|   tx_date|channel|         source_file|
+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----------+-------+--------------------+
|        NULL|2026-02-19 10:18:00|CARD300085|       NULL|7011|120.51|     USD|    AZE|Sumqayit|2026-02-19|   ECOM|s3a://bronze/land...|
|TX3287663670|2026-02-19 14:53:00|CARD300078|       NULL|6011| 16.09|     EUR|    DEU|  Munich|2026-02-19|   ECOM|s3a://bronze/land...|
|TX5112210761|2026-02-19 21:27:00|CARD300114| CUST100089|4829|285.67|     TRY|    DEU|  Munich|2026-02-19|   ECOM|s3a://bronze/land...|
|TX5328426280|2026-02-19 19:33:00|CARD300055| CUST100045|4111| 52.22|     TRY|    DEU|  Munich|2026-02-19|   ECOM|s3a://bronze/land...|
|TX4082141371|2026-02-19 08:07:00|CARD300063| CU

In [47]:
df_pos = spark.read.json("s3a://bronze/landing/pos/") \
    .filter(col("tx_date") == yesterday_str) \
    .withColumn("channel", lit("POS")) \
    .withColumn("source_file", input_file_name())

In [49]:
df_pos.show(5)

+------+----------+--------+-------------------+--------------------+----+------------+----------+-------+--------------------+
|amount|   card_no|currency|         event_time|            location| mcc|       tx_id|   tx_date|channel|         source_file|
+------+----------+--------+-------------------+--------------------+----+------------+----------+-------+--------------------+
| 84.07|CARD300115|     AZN|2026-02-19 21:13:00|{Ankara, TUR, T85...|5942|        NULL|2026-02-19|    POS|s3a://bronze/land...|
| 87.77|CARD300045|     AZN|2026-02-19 14:41:00|{Sumqayit, AZE, T...|5942|TX7805515830|2026-02-19|    POS|s3a://bronze/land...|
| 47.58|CARD300146|     AZN|2026-02-19 11:33:00|{Ganja, AZE, T560...|5732|TX3699808357|2026-02-19|    POS|s3a://bronze/land...|
| 29.11|CARD300040|     AZN|2026-02-19 11:29:00|{Sumqayit, AZE, T...|7011|TX2964152169|2026-02-19|    POS|s3a://bronze/land...|
|218.45|CARD300131|     AZN|2026-02-19 21:27:00|{Istanbul, TUR, T...|7995|TX6669987819|2026-02-19|    PO

In [50]:
df_atm.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- card_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)



In [51]:
df_ecom.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- card_no: string (nullable = true)
 |-- customer_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)



In [52]:
df_pos.printSchema()

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- terminal_id: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)



In [53]:
print("ATM :", df_atm.columns)
print("ECOM :", df_ecom.columns)
print("POS :", df_pos.columns)

ATM : ['tx_id', 'tx_ts', 'card_no', 'mcc', 'amount', 'currency', 'country', 'city', 'terminal_id', 'tx_date', 'channel', 'source_file']
ECOM : ['tx_id', 'tx_ts', 'card_no', 'customer_no', 'mcc', 'amount', 'currency', 'country', 'city', 'tx_date', 'channel', 'source_file']
POS : ['amount', 'card_no', 'currency', 'event_time', 'location', 'mcc', 'tx_id', 'tx_date', 'channel', 'source_file']


In [54]:
df_pos =df_pos.withColumn("city", col("location.city")) \
              .withColumn("country", col("location.country")) \
              .withColumn("terminal_id", col("location.terminal_id")) \
              .drop("location")

In [55]:
df_pos.printSchema()

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- terminal_id: string (nullable = true)



In [56]:
df_pos = df_pos.withColumnRenamed("event_time", "tx_ts")

In [57]:
df_pos.printSchema()

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- terminal_id: string (nullable = true)



In [58]:
df_pos = df_pos.withColumn("amount", col("amount").cast("double")) \
               .withColumn("mcc", col("mcc").cast("integer")) \
               .withColumn("tx_ts", col("tx_ts").cast("timestamp")) 

In [59]:
df_pos.printSchema()

root
 |-- amount: double (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- terminal_id: string (nullable = true)



In [60]:
df_transactions = df_ecom.unionByName(df_pos, allowMissingColumns=True) \
                         .unionByName(df_atm, allowMissingColumns=True)

In [61]:
df_transactions.count()

940

In [62]:
window_spec = Window.partitionBy("tx_id").orderBy(col("tx_ts").desc())

df_transactions = df_transactions.withColumn("rn", row_number().over(window_spec)) \
                                 .filter(col("rn") == 1) \
                                 .drop("rn")

In [63]:
df_final = df_transactions.withColumn("ingest_date", current_date())

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ingest_date") \
    .save("s3a://bronze/transactions")

In [64]:
from pyspark.sql.functions import when, concat

df_raw = spark.read.format("delta"). load("s3a://bronze/transactions")

df_dq = df_raw.withColumn("dq_reason", concat(
    when(col("tx_id").isNull(), lit("tx_id_null;")).otherwise(lit("")),
    when(col("tx_ts").isNull(), lit("tx_ts_null;")).otherwise(lit("")),
    when((col("amount") <= 0) | col("amount").isNull(), lit("amount_invalid;")).otherwise(lit("")),
    when(col("currency").isNull(), lit("currency_null;")).otherwise(lit(""))
))

df_good = df_dq.filter(col("dq_reason") == "").drop("dq_reason")
df_bad = df_dq.filter(col("dq_reason") != "")

df_good.write.format("delta").mode("overwrite").save("s3a://bronze/transactions_good")
df_bad.write.format("delta").mode("overwrite").save("s3a://bronze/transactions_bad")

print({df_good.count()})
print({df_bad.count()})

{937}
{2}


In [67]:
jdbc_url = "jdbc:postgresql://db:5432/postgres"
db_properties = {
    "user": "postgres",
    "password": "your_strong_password", 
    "driver": "org.postgresql.Driver"
}

In [68]:
tables = ["customers", "cards", "mcc_mapping", "fx_rate"]

for table_name in tables:
    print({table_name})
    
    df_temp = spark.read.jdbc(url=jdbc_url, table=f"public.{table_name}", properties=db_properties)
    
    (df_temp.write
        .format("delta")
        .mode("overwrite")
        .save(f"s3a://bronze/dims/{table_name}"))
    
    print({table_name})

{'customers'}
{'customers'}
{'cards'}
{'cards'}
{'mcc_mapping'}
{'mcc_mapping'}
{'fx_rate'}
{'fx_rate'}


In [69]:
spark.read.format("delta").load("s3a://bronze/dims/customers").show(5)

+-----------+------------------+------------+
|customer_no|         full_name|home_country|
+-----------+------------------+------------+
| CUST100000|  Vüqar Mustafayev|         AZE|
| CUST100001|      Lalə Abbasov|         TUR|
| CUST100002|     İlkin Qasımov|         DEU|
| CUST100003|Səbinə Şahverdiyev|         TUR|
| CUST100004|    Səbinə Əkbərov|         RUS|
+-----------+------------------+------------+
only showing top 5 rows



In [70]:
from pyspark.sql.functions import col, upper, trim, when, broadcast

df_transactions = spark.read.format("delta").load("s3a://bronze/transactions_good")
df_customers = spark.read.format("delta").load("s3a://bronze/dims/customers")
df_cards = spark.read.format("delta").load("s3a://bronze/dims/cards")
df_mcc = spark.read.format("delta").load("s3a://bronze/dims/mcc_mapping")
df_fx = spark.read.format("delta").load("s3a://bronze/dims/fx_rate")

In [72]:
df_transactions = df_transactions.withColumn("currency", upper(trim(col("currency")))) \
                                 .withColumn("country", upper(trim(col("country"))))

In [74]:
df_transactions.show(5)

+------------+-------------------+----------+-----------+----+------+--------+-------+----------+----------+-------+--------------------+-----------+-----------+
|       tx_id|              tx_ts|   card_no|customer_no| mcc|amount|currency|country|      city|   tx_date|channel|         source_file|terminal_id|ingest_date|
+------------+-------------------+----------+-----------+----+------+--------+-------+----------+----------+-------+--------------------+-----------+-----------+
|TX0023325730|2026-02-19 19:36:00|CARD300015|       NULL|4829|566.62|     AZN|    GBR|Manchester|2026-02-19|    POS|s3a://bronze/land...|    T250580| 2026-03-01|
|TX0030309941|2026-02-19 23:41:00|CARD300004|       NULL|4111|  59.4|     AZN|    AZE|     Ganja|2026-02-19|    POS|s3a://bronze/land...|    T435794| 2026-03-01|
|TX0035547444|2026-02-19 10:21:00|CARD300144|       NULL|5732| 39.73|     USD|    DEU|    Berlin|2026-02-19|    POS|s3a://bronze/land...|    T516526| 2026-03-01|
|TX0044852080|2026-02-19 20:

In [75]:
df_cards.printSchema()

root
 |-- card_no: string (nullable = true)
 |-- customer_no: string (nullable = true)



In [76]:
df_transactions.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- card_no: string (nullable = true)
 |-- customer_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- ingest_date: date (nullable = true)



In [77]:
from pyspark.sql.functions import coalesce

df_silver = df_transactions.join(
    df_cards.select("card_no", col("customer_no").alias("cust_from_cards")), 
    on="card_no", 
    how="left"
)

df_silver = df_silver.withColumn(
    "customer_no", 
    coalesce(df_silver["customer_no"], df_silver["cust_from_cards"])
).drop("cust_from_cards")

df_silver = df_silver.join(
    df_customers.select("customer_no", "home_country"), 
    on="customer_no", 
    how="left"
)

df_silver.select("tx_id", "card_no", "customer_no", "home_country").show(5)

+------------+----------+-----------+------------+
|       tx_id|   card_no|customer_no|home_country|
+------------+----------+-----------+------------+
|TX0023325730|CARD300015| CUST100012|         AZE|
|TX0030309941|CARD300004| CUST100003|         TUR|
|TX0035547444|CARD300144| CUST100111|         AZE|
|TX0044852080|CARD300121| CUST100094|         AZE|
|TX0062583419|CARD300075| CUST100061|         AZE|
+------------+----------+-----------+------------+
only showing top 5 rows



In [78]:
from pyspark.sql.functions import col, broadcast, to_date, round

df_silver = df_transactions.join(
    df_cards.select("card_no", col("customer_no").alias("cust_from_cards")), 
    on="card_no", 
    how="left"
)

In [79]:
df_silver = df_silver.withColumn(
    "customer_no", 
    coalesce(df_silver["customer_no"], df_silver["cust_from_cards"])
).drop("cust_from_cards")

df_silver = df_silver.join(
    df_customers.select("customer_no", "home_country"), 
    on="customer_no", 
    how="left"
)

In [80]:
df_silver = df_silver.join(broadcast(df_mcc), on="mcc", how="left")

In [81]:
df_silver = df_silver.withColumn("tx_date", to_date(col("tx_ts")))

In [82]:
df_fx_filtered = df_fx.select(col("fx_date"), col("currency").alias("fx_currency"), col("rate_to_azn"))

In [83]:
df_silver = df_silver.join(
    df_fx_filtered, 
    (df_silver["tx_date"] == df_fx_filtered["fx_date"]) & (df_silver["currency"] == df_fx_filtered["fx_currency"]), 
    how="left"
).drop("fx_date", "fx_currency")

In [85]:
df_silver = df_silver.withColumn("amount_azn", round(col("amount") * col("rate_to_azn"), 2)) \
                     .withColumn("is_foreign", when(col("country") != col("home_country"), 1).otherwise(0)) \
                     .withColumn("is_high_risk", when(col("risk_weight") >= 8, 1).otherwise(0))

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("s3a://silver/transactions"))

In [86]:
from pyspark.sql.functions import count, sum, count_distinct, avg

df_gold = df_silver.groupBy("customer_no", "tx_date").agg(
    count("tx_id").alias("txn_count"),
    sum("amount_azn").alias("total_amount_azn"),
    sum("is_foreign").alias("foreign_txn_count"),
    sum("is_high_risk").alias("high_risk_txn_count"),
    count_distinct("country").alias("distinct_countries"),
    avg("amount_azn").alias("avg_amount_azn"),
    avg("risk_weight").alias("avg_risk_weight")
)

In [87]:
df_gold = df_gold.withColumn(
    "risk_score", 
    (col("foreign_txn_count") * 2) + (col("high_risk_txn_count") * 3) + col("avg_risk_weight")
)

df_gold = df_gold.withColumn(
    "risk_level",
    when(col("risk_score") < 10, lit("LOW"))
    .when((col("risk_score") >= 10) & (col("risk_score") < 20), lit("MEDIUM"))
    .otherwise(lit("HIGH"))
)

In [88]:
df_gold.select("customer_no", "tx_date", "risk_score", "risk_level").show(5)

+-----------+----------+------------------+----------+
|customer_no|   tx_date|        risk_score|risk_level|
+-----------+----------+------------------+----------+
| CUST100094|2026-02-19|              17.0|    MEDIUM|
| CUST100061|2026-02-19|             12.75|    MEDIUM|
| CUST100070|2026-02-19|              13.0|    MEDIUM|
| CUST100032|2026-02-19| 5.666666666666667|       LOW|
| CUST100056|2026-02-19|15.727272727272727|    MEDIUM|
+-----------+----------+------------------+----------+
only showing top 5 rows



In [89]:
df_gold_final = df_gold.coalesce(1)

(df_gold_final.write
    .format("delta")
    .mode("overwrite")
    .save("s3a://gold/customer_daily_risk"))

In [90]:
(df_gold_final.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.customer_daily_risk")
    .option("user", db_properties["user"])
    .option("password", db_properties["password"])
    .option("driver", db_properties["driver"])
    .option("batchsize", "1000")
    .mode("append")
    .save())